# Weather Type Classification — Applying Probability & Statistics to Evaluate an ML Model

**Course:** MSIS 2026–27 · **Dataset:** [Kaggle – Weather Type Classification](https://www.kaggle.com/datasets/nikhil7280/weather-type-classification)

**Research question:** *Does the improved model (Random Forest) perform better than the baseline (Decision Tree), or could the observed difference be due to variation in the test sample?*

| Step | What this notebook does |
|---|---|
| 1 | Load the Kaggle dataset (auto-download, or upload the CSV) |
| 2 | Inspect & clean: missing values, duplicates, implausible values |
| 3 | Exploratory data analysis: descriptive statistics, distributions, correlation |
| 4 | Stratified 80/20 train–test split (same test set for both models) |
| 5 | Baseline **Decision Tree** and improved **Random Forest** (tuned with 5-fold CV on training data only) |
| 6 | Evaluation: accuracy, balanced accuracy, precision, recall, macro F1, confusion matrices |
| 7 | Statistics: paired bootstrap (95% CIs), **McNemar's test** (α = 0.05) |
| 8 | Results table, decision and limitations |
| 9 | **Deployment:** saved model bundle + interactive *Weather Check* web app (Gradio) |

> **Run:** `Runtime → Run all`. Everything is seeded (`SEED = 42`), so results are reproducible.

## 0. Setup

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score)

warnings.filterwarnings("ignore")
SEED = 42                 # global random seed -> reproducible split, models and bootstrap
N_BOOT = 2000             # number of bootstrap resamples
ALPHA = 0.05              # significance level
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
os.makedirs("outputs", exist_ok=True)   # figures/tables saved here for the report
print("Setup complete.")

## 1. Dataset collection
The loader tries, in order: (a) a CSV already in the working folder, (b) automatic download with `kagglehub` (pre-installed on Colab), (c) a manual upload prompt.
If (b) fails, download `weather_classification_data.csv` from the Kaggle link above and upload it when prompted.

In [ ]:
CSV_NAME = "weather_classification_data.csv"
TARGET = "Weather Type"

def load_dataset():
    # (a) local file
    local = glob.glob(CSV_NAME) + glob.glob("*weather*.csv")
    if local:
        print("Loaded local file:", local[0]); return pd.read_csv(local[0])
    # (b) Kaggle download
    try:
        import kagglehub
        path = kagglehub.dataset_download("nikhil7280/weather-type-classification")
        files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
        print("Downloaded from Kaggle:", files[0]); return pd.read_csv(files[0])
    except Exception as e:
        print("Automatic Kaggle download failed ->", e)
    # (c) manual upload (Colab)
    from google.colab import files
    print("Please upload weather_classification_data.csv")
    uploaded = files.upload()
    return pd.read_csv(next(iter(uploaded)))

df_raw = load_dataset()
print("Shape:", df_raw.shape)
df_raw.head()

### 1.1 Inspect structure (verify columns, types and labels against the dataset description)

In [ ]:
print("Columns:", list(df_raw.columns), "\n")
print(df_raw.dtypes, "\n")
print("Missing values per column:\n", df_raw.isna().sum(), "\n")
print("Exact duplicate rows:", df_raw.duplicated().sum(), "\n")
print("Class counts:\n", df_raw[TARGET].value_counts())

## 2. Data preprocessing
* **Duplicates** are removed before splitting, so an identical row cannot appear in both train and test (leakage).
* **Implausible values:** humidity and precipitation are percentages, so values > 100 are physically impossible. They are set to missing and later imputed with the **training-set median** (inside the pipeline, so no test information leaks).
* Categorical predictors (e.g. Cloud Cover, Season, Location) are one-hot encoded inside the pipeline. Scaling is not needed for tree-based models.

In [ ]:
df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)} (removed {len(df_raw) - len(df)})")

pct_cols = [c for c in df.columns if ("Humidity" in c) or ("Precipitation" in c)]
for c in pct_cols:
    bad = (df[c] > 100) | (df[c] < 0)
    print(f"{c}: {bad.sum()} implausible values (outside 0–100) -> set to NaN")
    df.loc[bad, c] = np.nan

X = df.drop(columns=[TARGET])
y = df[TARGET]
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(exclude=np.number).columns.tolist()
print("\nNumeric features:", num_cols)
print("Categorical features:", cat_cols)

## 3. Exploratory data analysis
Descriptive statistics (mean, median, variance, standard deviation) summarise the **distribution** of each feature; the class bar chart checks **class balance**; box plots and the correlation matrix show how features relate to weather type and to each other.

In [ ]:
desc = pd.DataFrame({
    "mean": df[num_cols].mean(), "median": df[num_cols].median(),
    "variance": df[num_cols].var(), "std": df[num_cols].std(),
    "min": df[num_cols].min(), "max": df[num_cols].max(),
    "skewness": df[num_cols].skew()}).round(3)
desc.to_csv("outputs/descriptive_statistics.csv")
desc

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = y.value_counts()
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="Set2")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v}\n({v/len(y):.1%})", ha="center", va="bottom", fontsize=9)
ax.set_title("Class distribution"); ax.set_ylabel("Count"); ax.set_ylim(0, counts.max() * 1.2)
plt.tight_layout(); plt.savefig("outputs/class_distribution.png", dpi=150); plt.show()

In [ ]:
n = len(num_cols); cols = 3; rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(df[c].dropna(), kde=True, ax=ax, color="steelblue"); ax.set_title(c)
for ax in axes.ravel()[n:]: ax.axis("off")
plt.suptitle("Feature distributions", y=1.01); plt.tight_layout()
plt.savefig("outputs/histograms.png", dpi=150); plt.show()

fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))
for ax, c in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x=TARGET, y=c, ax=ax, palette="Set2"); ax.set_title(f"{c} by weather type")
for ax in axes.ravel()[n:]: ax.axis("off")
plt.tight_layout(); plt.savefig("outputs/boxplots_by_class.png", dpi=150); plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Pearson correlation between numeric features")
plt.tight_layout(); plt.savefig("outputs/correlation_heatmap.png", dpi=150); plt.show()

## 4. Train–test split
Stratified 80/20 split: each weather class keeps the same proportion in both sets. **Both models are evaluated on exactly the same test rows**, which makes the later comparison *paired*.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
print("Train:", X_train.shape, " Test:", X_test.shape)
split_props = pd.DataFrame({"train %": y_train.value_counts(normalize=True).round(3),
                            "test %": y_test.value_counts(normalize=True).round(3)})
split_props

Preprocessing is fitted **on the training data only** (inside each pipeline).

In [ ]:
def make_preprocessor():
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

## 5A. Baseline model — Decision Tree
A single tree with default settings: it splits on the feature/threshold that most reduces **Gini impurity**. Interpretable, but prone to overfitting (high variance).

In [ ]:
dt_model = Pipeline([("prep", make_preprocessor()),
                     ("clf", DecisionTreeClassifier(random_state=SEED))])
dt_model.fit(X_train, y_train)
print("Decision Tree trained. Depth:", dt_model.named_steps["clf"].get_depth(),
      "| Leaves:", dt_model.named_steps["clf"].get_n_leaves())

## 5B. Improved model — Random Forest
An ensemble of trees, each trained on a **bootstrap sample** of the training data with a random subset of features; predictions are combined by **majority vote**, which reduces variance.
Hyper-parameters are tuned with **5-fold stratified cross-validation on the training set only** (the test set stays untouched).

In [ ]:
rf_pipe = Pipeline([("prep", make_preprocessor()),
                    ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1))])
param_grid = {"clf__n_estimators": [200, 400],
              "clf__max_depth": [None, 20],
              "clf__min_samples_leaf": [1, 2]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
search = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1)
search.fit(X_train, y_train)
rf_model = search.best_estimator_
print("Best RF parameters:", search.best_params_)
print(f"Mean 5-fold CV macro F1 (training data): {search.best_score_:.4f}")

## 6. Performance evaluation on the same held-out test set

In [ ]:
y_pred_dt = dt_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)
labels = sorted(y.unique())

def metrics(y_true, y_pred):
    return {"Accuracy": accuracy_score(y_true, y_pred),
            "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
            "Macro precision": precision_score(y_true, y_pred, average="macro"),
            "Macro recall": recall_score(y_true, y_pred, average="macro"),
            "Macro F1": f1_score(y_true, y_pred, average="macro")}

point = pd.DataFrame({"Decision Tree": metrics(y_test, y_pred_dt),
                      "Random Forest": metrics(y_test, y_pred_rf)})
point["Difference (RF − DT)"] = point["Random Forest"] - point["Decision Tree"]
point.round(4).to_csv("outputs/point_metrics.csv")
point.round(4)

In [ ]:
print("=== Decision Tree ===\n", classification_report(y_test, y_pred_dt, digits=4))
print("=== Random Forest ===\n", classification_report(y_test, y_pred_rf, digits=4))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, pred, name in zip(axes, [y_pred_dt, y_pred_rf], ["Decision Tree", "Random Forest"]):
    ConfusionMatrixDisplay(confusion_matrix(y_test, pred, labels=labels),
                           display_labels=labels).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name} — confusion matrix")
plt.tight_layout(); plt.savefig("outputs/confusion_matrices.png", dpi=150); plt.show()

## 7. Statistical analysis

### 7.1 Paired bootstrap resampling → 95% confidence intervals
The test set is only **one sample** of possible observations; a different sample would give slightly different scores. To estimate this sampling variability we:

1. Draw `n_test` row indices **with replacement** from the test set (one bootstrap replicate).
2. Compute accuracy and macro F1 for **both** models on the *same* resampled rows (this keeps the pairing).
3. Record the paired difference RF − DT.
4. Repeat `N_BOOT = 2000` times; the 2.5th and 97.5th percentiles form the **95% percentile CI**.

If the CI of the *difference* excludes 0, the improvement is unlikely to be explained by test-sample variation alone.

In [ ]:
classes = np.array(labels)
yt  = np.searchsorted(classes, y_test.to_numpy())
pdt = np.searchsorted(classes, y_pred_dt)
prf = np.searchsorted(classes, y_pred_rf)
K, n_test = len(classes), len(yt)

def macro_f1_fast(t, p):
    # Macro F1 from a confusion matrix (fast version of sklearn's f1_score(average='macro'))
    cm = np.bincount(t * K + p, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    denom = 2 * tp + fp + fn
    f1 = np.where(denom > 0, 2 * tp / np.maximum(denom, 1), 0.0)
    return f1.mean()

assert np.isclose(macro_f1_fast(yt, prf), f1_score(y_test, y_pred_rf, average="macro"))

rng = np.random.default_rng(SEED)
correct_dt, correct_rf = (pdt == yt), (prf == yt)
boot = {k: np.empty(N_BOOT) for k in ["acc_dt", "acc_rf", "f1_dt", "f1_rf"]}
for b in range(N_BOOT):
    idx = rng.integers(0, n_test, n_test)          # sample WITH replacement
    boot["acc_dt"][b] = correct_dt[idx].mean()
    boot["acc_rf"][b] = correct_rf[idx].mean()
    boot["f1_dt"][b]  = macro_f1_fast(yt[idx], pdt[idx])
    boot["f1_rf"][b]  = macro_f1_fast(yt[idx], prf[idx])
boot["acc_diff"] = boot["acc_rf"] - boot["acc_dt"]
boot["f1_diff"]  = boot["f1_rf"] - boot["f1_dt"]

def ci(a): return np.percentile(a, [2.5, 97.5])

observed = {"acc_dt": correct_dt.mean(), "acc_rf": correct_rf.mean(),
            "f1_dt": macro_f1_fast(yt, pdt), "f1_rf": macro_f1_fast(yt, prf)}
observed["acc_diff"] = observed["acc_rf"] - observed["acc_dt"]
observed["f1_diff"]  = observed["f1_rf"] - observed["f1_dt"]

rows = []
for key, name in [("acc_dt", "Accuracy – Decision Tree"), ("acc_rf", "Accuracy – Random Forest"),
                  ("acc_diff", "Accuracy difference (RF − DT)"),
                  ("f1_dt", "Macro F1 – Decision Tree"), ("f1_rf", "Macro F1 – Random Forest"),
                  ("f1_diff", "Macro F1 difference (RF − DT)")]:
    lo, hi = ci(boot[key])
    rows.append({"Measure": name, "Observed": observed[key], "Bootstrap mean": boot[key].mean(),
                 "Bootstrap SE": boot[key].std(ddof=1), "95% CI lower": lo, "95% CI upper": hi,
                 "CI width": hi - lo})
ci_table = pd.DataFrame(rows).set_index("Measure").round(4)
ci_table.to_csv("outputs/bootstrap_confidence_intervals.csv")
ci_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (m, title) in zip(axes, [("acc", "Accuracy"), ("f1", "Macro F1")]):
    sns.histplot(boot[f"{m}_dt"], ax=ax, color="tab:orange", label="Decision Tree", kde=True, stat="density")
    sns.histplot(boot[f"{m}_rf"], ax=ax, color="tab:blue", label="Random Forest", kde=True, stat="density")
    ax.set_title(f"Bootstrap distribution of {title} (B = {N_BOOT})"); ax.legend()
plt.tight_layout(); plt.savefig("outputs/bootstrap_distributions.png", dpi=150); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (m, title) in zip(axes, [("acc", "Accuracy"), ("f1", "Macro F1")]):
    d = boot[f"{m}_diff"]; lo, hi = ci(d)
    sns.histplot(d, ax=ax, kde=True, color="tab:green", stat="density")
    ax.axvline(0, color="red", ls="--", label="No difference (0)")
    ax.axvline(lo, color="black", ls=":", label="95% CI"); ax.axvline(hi, color="black", ls=":")
    ax.set_title(f"Paired bootstrap: {title} difference (RF − DT)"); ax.legend()
plt.tight_layout(); plt.savefig("outputs/bootstrap_differences.png", dpi=150); plt.show()

### 7.2 Hypothesis test — McNemar's test (paired correctness)
Each test observation gives a pair of outcomes (DT correct?, RF correct?). Only the **discordant** pairs carry information about which model is better:

* **b** = DT correct, RF wrong
* **c** = DT wrong, RF correct

**H₀:** both models have the same probability of a correct prediction, i.e. P(b) = P(c).
**H₁:** the probabilities differ (two-sided).  α = 0.05.

Under H₀, each discordant pair is equally likely to fall in b or c, so **c ~ Binomial(b + c, 0.5)** → exact test.
The large-sample version uses the χ² statistic with continuity correction: χ² = (|b − c| − 1)² / (b + c), df = 1.

In [ ]:
both_right = int(np.sum(correct_dt & correct_rf))
b = int(np.sum(correct_dt & ~correct_rf))     # DT right, RF wrong
c = int(np.sum(~correct_dt & correct_rf))     # DT wrong, RF right
both_wrong = int(np.sum(~correct_dt & ~correct_rf))

table = pd.DataFrame([[both_right, b], [c, both_wrong]],
                     index=["DT correct", "DT wrong"], columns=["RF correct", "RF wrong"])
print("McNemar contingency table:"); display(table)

n_disc = b + c
if n_disc == 0:
    chi2_stat, p_chi2, p_exact, p_one = 0.0, 1.0, 1.0, 1.0
    print("No discordant pairs: the models make identical correct/incorrect decisions.")
else:
    chi2_stat = (abs(b - c) - 1) ** 2 / n_disc
    p_chi2 = stats.chi2.sf(chi2_stat, df=1)
    p_exact = stats.binomtest(c, n_disc, 0.5, alternative="two-sided").pvalue
    p_one = stats.binomtest(c, n_disc, 0.5, alternative="greater").pvalue  # H1: RF better

# Exact test is preferred when discordant pairs are few (< 25); otherwise chi-square is standard
use_exact = n_disc < 25
p_main = p_exact if use_exact else p_chi2
decision = "Reject H0" if p_main < ALPHA else "Fail to reject H0"

print(f"Discordant pairs b = {b}, c = {c}  (b + c = {n_disc})")
print(f"McNemar chi-square (continuity-corrected) = {chi2_stat:.4f}, p = {p_chi2:.4g}")
print(f"McNemar exact binomial (two-sided)          p = {p_exact:.4g}")
print(f"Supplementary one-sided exact (H1: RF better) p = {p_one:.4g}")
print(f"\nPrimary test: {'exact' if use_exact else 'chi-square'}  ->  Decision at α = {ALPHA}: {decision}")

### 7.3 Supplementary: what does the improvement look like across different splits?
The test above conditions on **one** split. As a robustness check, both models are re-trained on several different random 80/20 splits (RF uses the tuned parameters) to see whether RF wins consistently. This is descriptive only — overlapping training sets make these runs correlated, so no p-value is computed from them.

In [ ]:
best_rf_params = {k.replace("clf__", ""): v for k, v in search.best_params_.items()}
rep = []
for s in range(10):
    Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=100 + s)
    dt_s = Pipeline([("prep", make_preprocessor()), ("clf", DecisionTreeClassifier(random_state=SEED))]).fit(Xa, ya)
    rf_s = Pipeline([("prep", make_preprocessor()),
                     ("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1, **best_rf_params))]).fit(Xa, ya)
    rep.append({"split": s, "DT macro F1": f1_score(yb, dt_s.predict(Xb), average="macro"),
                "RF macro F1": f1_score(yb, rf_s.predict(Xb), average="macro")})
rep = pd.DataFrame(rep); rep["RF − DT"] = rep["RF macro F1"] - rep["DT macro F1"]
print(f"RF better in {(rep['RF − DT'] > 0).sum()} of {len(rep)} splits; "
      f"mean difference = {rep['RF − DT'].mean():.4f} (sd {rep['RF − DT'].std():.4f})")
rep.round(4)

### 7.4 Random Forest feature importance (to discuss possible label-encoding features)

In [ ]:
feat_names = rf_model.named_steps["prep"].get_feature_names_out()
imp = pd.Series(rf_model.named_steps["clf"].feature_importances_, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(8, 6)); imp.head(15)[::-1].plot.barh(color="tab:blue")
plt.title("Top 15 Random Forest feature importances (mean decrease in impurity)")
plt.tight_layout(); plt.savefig("outputs/feature_importance.png", dpi=150); plt.show()
imp.head(15).round(4)

## 8. Final results table and decision

In [ ]:
def fmt_ci(key):
    lo, hi = ci(boot[key]); return f"[{lo:.4f}, {hi:.4f}]"

results = pd.DataFrame({
    "Decision Tree": [f"{observed['acc_dt']:.4f}", f"{observed['f1_dt']:.4f}", fmt_ci("acc_dt"), fmt_ci("f1_dt")],
    "Random Forest": [f"{observed['acc_rf']:.4f}", f"{observed['f1_rf']:.4f}", fmt_ci("acc_rf"), fmt_ci("f1_rf")],
    "Difference (RF − DT)": [f"{observed['acc_diff']:+.4f}", f"{observed['f1_diff']:+.4f}",
                             fmt_ci("acc_diff"), fmt_ci("f1_diff")]},
    index=["Accuracy", "Macro F1", "95% CI – Accuracy", "95% CI – Macro F1"])
results.to_csv("outputs/final_results_table.csv")
display(results)

print(f"McNemar's test: statistic = {chi2_stat:.4f} | p-value = {p_main:.4g} "
      f"({'exact' if use_exact else 'chi-square'}) | Decision at α = {ALPHA}: {decision}")

In [ ]:
acc_lo, acc_hi = ci(boot["acc_diff"]); f1_lo, f1_hi = ci(boot["f1_diff"])
better = observed["acc_diff"] > 0
lines = [
    f"Random Forest accuracy {observed['acc_rf']:.4f} vs Decision Tree {observed['acc_dt']:.4f} "
    f"(difference {observed['acc_diff']:+.4f}, 95% CI [{acc_lo:+.4f}, {acc_hi:+.4f}]).",
    f"Macro F1 difference {observed['f1_diff']:+.4f}, 95% CI [{f1_lo:+.4f}, {f1_hi:+.4f}].",
    f"McNemar: b = {b} (only DT right), c = {c} (only RF right), p = {p_main:.4g} -> {decision}."]
if decision == "Reject H0" and better and acc_lo > 0 and f1_lo > 0:
    lines.append("CONCLUSION: The Random Forest's improvement is statistically significant. Both paired CIs "
                 "exclude 0 and McNemar's test rejects H0, so the gain is unlikely to be explained by "
                 "variation in the test sample alone.")
elif decision == "Reject H0" and not better:
    lines.append("CONCLUSION: The models differ significantly, but in favour of the Decision Tree.")
elif decision == "Reject H0":
    lines.append("CONCLUSION: McNemar's test rejects H0 for accuracy, but at least one paired CI includes 0; "
                 "the evidence is mixed across metrics and should be reported with caution.")
else:
    lines.append("CONCLUSION: We fail to reject H0. The observed difference is small enough that it could be "
                 "due to test-sample variation. This does NOT prove the models are identical — only that this "
                 "test set does not provide enough evidence of a difference.")
conclusion = "\n".join(lines)
print(conclusion)
with open("outputs/conclusion.txt", "w") as f: f.write(conclusion)

## 9. Where probability and statistics are used

| Stage | Technique | Purpose |
|---|---|---|
| EDA | Mean, median, variance, standard deviation, skewness, correlation | Describe feature distributions and relationships |
| Preprocessing | Median imputation | Robust estimate of a central value for missing/implausible entries |
| Train–test split | Stratified random sampling | Keep class proportions equal in train and test |
| Random Forest | Bootstrap sampling + random feature subsets, majority vote | Reduce variance of individual trees |
| Model tuning | k-fold cross-validation | Estimate generalisation without touching the test set |
| Evaluation | Accuracy, precision, recall, F1 as empirical probabilities | Estimate P(correct), P(true class \| predicted class), etc. |
| Uncertainty | Paired bootstrap, 95% percentile CI, standard error | Quantify how much scores vary across possible test samples |
| Hypothesis test | McNemar's test (binomial / χ² distribution), α = 0.05 | Decide whether the difference in correctness is significant |

## 10. Limitations
* **Classification, not forecasting:** the dataset labels current conditions; it cannot predict tomorrow's weather without a time-indexed target.
* **Synthetic / unrealistic values:** the dataset appears to be generated and contains impossible values (e.g. humidity or precipitation > 100%), which were treated as missing. Real-world performance may differ.
* **Possible label leakage:** features such as precipitation, cloud cover, visibility and UV index may nearly encode the label (see feature importance), which can inflate both models' scores.
* **Single split:** McNemar's test and the bootstrap condition on one train–test split; they capture test-sample variation but not training-set variation (Section 7.3 gives a descriptive check).
* **Baseline not tuned:** the Decision Tree uses default settings while the Random Forest is tuned, so part of the gap reflects tuning effort as well as the algorithm.
* **Multiple metrics:** several metrics are reported; the McNemar test (accuracy-based) is the pre-specified primary test.

## 11. Model deployment — "Weather Check" demo app
This section turns the trained models into a small web app (built with **Gradio**), in the style of a clinical-AI demo:

* **Enter observations** by hand, or **try a real example** from the held-out test set (rows neither model saw in training).
* The app shows the **predicted weather type and its probability**, the **probability of every class**, and a **"Show why"** panel per class that tells you which readings drove the decision.
* It also shows the model's **reliability from Section 7**: test accuracy with its 95% bootstrap confidence interval, and what the other model predicts.

**How "Show why" works (local occlusion explanation):** each reading is, one at a time, replaced by a typical training value (median, or most common category) and the change in the class probability is measured. A large drop means that reading pushed the model towards that class.

In [ ]:
import sys, html, joblib

# ---- save the trained models + everything the app needs (this file is the deployable artefact) ----
FEATURES = list(X.columns)
BASELINE_ROW = {c: (X_train[c].median() if c in num_cols else X_train[c].mode()[0]) for c in FEATURES}
RANGES = {c: (float(X_train[c].min()), float(X_train[c].max())) for c in num_cols}
CATEGORIES = {c: sorted(X_train[c].dropna().unique().tolist()) for c in cat_cols}
MODELS = {"Random Forest (improved)": rf_model, "Decision Tree (baseline)": dt_model}
RELIABILITY = {"Random Forest (improved)": (observed["acc_rf"], *ci(boot["acc_rf"])),
               "Decision Tree (baseline)": (observed["acc_dt"], *ci(boot["acc_dt"]))}
joblib.dump({"models": MODELS, "features": FEATURES, "baseline_row": BASELINE_ROW, "ranges": RANGES,
             "categories": CATEGORIES, "reliability": RELIABILITY, "pct_cols": pct_cols},
            "outputs/weather_check_models.joblib")
print("Saved deployable model bundle -> outputs/weather_check_models.joblib")

ICON = {"Sunny": "☀️", "Cloudy": "☁️", "Rainy": "🌧️", "Snowy": "❄️"}
DESC = {"Sunny": "clear skies and strong sunshine", "Cloudy": "cloud cover with little or no precipitation",
        "Rainy": "precipitation falling as rain", "Snowy": "cold conditions with precipitation as snow"}

def make_row(values):
    # Build a one-row DataFrame from form values, applying the same cleaning rule as training
    row = {}
    for c, v in zip(FEATURES, values):
        if c in num_cols:
            row[c] = np.nan if v is None or v == "" else float(v)
        else:
            row[c] = v
    return pd.DataFrame([row], columns=FEATURES)

def input_warnings(row):
    notes = []
    for c in num_cols:
        v = row[c].iloc[0]
        if pd.isna(v):
            notes.append(f"<b>{html.escape(c)}</b> is missing; the training median is used instead.")
            continue
        if c in pct_cols and not (0 <= v <= 100):
            notes.append(f"<b>{html.escape(c)}</b> = {v:g} is outside 0–100%, so it is treated as missing (as in training).")
            row.loc[row.index[0], c] = np.nan
            continue
        lo, hi = RANGES[c]
        if v < lo or v > hi:
            notes.append(f"<b>{html.escape(c)}</b> = {v:g} is outside the training range [{lo:g}, {hi:g}]; "
                         "the prediction is an extrapolation.")
    return row, notes

def explain(model, row):
    # Occlusion: replace one feature at a time by its typical training value and measure the change
    variants = [row] + [row.assign(**{c: BASELINE_ROW[c]}) for c in FEATURES]
    P = model.predict_proba(pd.concat(variants, ignore_index=True))
    return model.classes_, P[0], pd.DataFrame(P[0] - P[1:], index=FEATURES, columns=model.classes_)

# ---------- HTML rendering (dark theme, inline styles so it works inside Gradio and Jupyter) ----------
FONT = "font-family:Inter,Segoe UI,Helvetica,Arial,sans-serif;"
CARD = ("background:#1b1b1e;border:1px solid #34343a;border-radius:12px;padding:18px 20px;"
        "margin:12px 0;color:#e7e7ea;" + FONT)
ORANGE, MUTED = "#fb923c", "#a1a1aa"

def header_html():
    return (f'<div style="{FONT}color:#f4f4f5;padding:4px 2px">'
            '<div style="font-size:30px;font-weight:800">Weather Check</div>'
            f'<div style="color:{MUTED};margin:4px 0 12px">Give it a set of weather readings. The model says '
            'which weather type it detects and shows which readings led it there.</div>'
            '<div style="background:#3b1414;border:1px solid #7f1d1d;color:#fca5a5;border-radius:8px;'
            'padding:10px 14px;font-size:13px">Classroom prototype trained on the synthetic Kaggle '
            '<i>Weather Type Classification</i> dataset. It classifies current conditions; it is not a '
            'forecast and must not be used for safety decisions.</div></div>')

def fmt_value(c, v):
    if c in num_cols:
        return "missing" if pd.isna(v) else f"{v:g}"
    return html.escape(str(v))

def why_panel(cls, deltas, row):
    d = deltas[cls].sort_values(key=np.abs, ascending=False).head(5)
    scale = max(d.abs().max(), 1e-9)
    items = []
    for feat, val in d.items():
        w = abs(val) / scale * 48
        col = "#22c55e" if val > 0 else "#ef4444"
        side = f"left:50%;width:{w:.1f}%" if val > 0 else f"left:{50 - w:.1f}%;width:{w:.1f}%"
        verb = "pushes towards" if val > 0 else "pushes away from"
        items.append(
            '<div style="display:grid;grid-template-columns:200px 1fr 70px;gap:10px;align-items:center;margin:6px 0">'
            f'<div style="font-size:13px">{html.escape(feat)} = <b>{fmt_value(feat, row[feat].iloc[0])}</b></div>'
            '<div style="position:relative;height:10px;background:#2a2a2f;border-radius:5px">'
            '<div style="position:absolute;left:50%;top:-3px;width:1px;height:16px;background:#71717a"></div>'
            f'<div style="position:absolute;{side};height:10px;background:{col};border-radius:5px"></div></div>'
            f'<div style="font-size:12px;color:{col};text-align:right" title="{verb} {cls}">'
            f'{val * 100:+.1f} pts</div></div>')
    if d.abs().max() < 1e-6:
        items = [f'<div style="font-size:13px;color:{MUTED}">No single reading changes this probability on its own.</div>']
    return (f'<div style="margin:8px 0 4px;padding:10px 12px;background:#141416;border-radius:8px">'
            f'<div style="font-size:12px;color:{MUTED};margin-bottom:4px">Top readings for <b>{cls}</b>. '
            'Green = pushes towards this class, red = pushes away. Points = change in probability when the '
            'reading is replaced by a typical training value.</div>' + "".join(items) + "</div>")

def render_result(row, model_name, actual=None, source="Your observation"):
    row, notes = input_warnings(row.copy())
    model = MODELS[model_name]
    classes, probs, deltas = explain(model, row)
    k = int(np.argmax(probs)); pred = classes[k]

    other_name = [m for m in MODELS if m != model_name][0]
    other_p = MODELS[other_name].predict_proba(row)[0]
    other_pred = MODELS[other_name].classes_[int(np.argmax(other_p))]

    out = [f'<div style="{CARD}"><div style="font-size:18px;font-weight:700;margin-bottom:12px">'
           f'Result for {html.escape(source)}</div>'
           f'<div style="background:#3a1f0c;border:1px solid #7c2d12;color:{ORANGE};border-radius:8px;'
           f'padding:14px 16px;font-size:21px;font-weight:600">{ICON.get(pred, "")} The {model_name.split(" (")[0]} '
           f'predicts <b>{html.escape(str(pred))}</b> ({probs[k]:.0%}).</div>']
    sub = []
    if actual is not None:
        ok = (actual == pred)
        sub.append(f'Recorded label: <b>{html.escape(str(actual))}</b> '
                   f'<span style="color:{"#22c55e" if ok else "#ef4444"}">'
                   f'{"✓ correct" if ok else "✗ incorrect"}</span>.')
    agree = "agrees" if other_pred == pred else "disagrees"
    sub.append(f'{other_name} {agree}: it predicts <b>{html.escape(str(other_pred))}</b> ({other_p.max():.0%}).')
    acc, lo, hi = RELIABILITY[model_name]
    sub.append(f'Reliability on held-out test data: correct {acc:.1%} of the time '
               f'(95% bootstrap CI {lo:.1%} – {hi:.1%}).')
    out.append(f'<div style="color:{MUTED};font-size:13px;margin-top:10px;line-height:1.6">' + "<br>".join(sub) + "</div>")
    if notes:
        out.append('<div style="margin-top:10px;background:#3a300c;border:1px solid #854d0e;color:#fde68a;'
                   'border-radius:8px;padding:8px 12px;font-size:13px">' + "<br>".join(notes) + "</div>")
    out.append("</div>")

    out.append(f'<div style="{CARD}"><div style="font-size:18px;font-weight:700">What the model checked for</div>'
               f'<div style="color:{MUTED};font-size:13px;margin:6px 0 10px">Each bar is how likely the model thinks '
               'that weather type is. The highest bar is the prediction. Open <b>Show why</b> to see which readings '
               'drove each probability.</div>')
    for i, cls in enumerate(classes):
        fill = ORANGE if i == k else "#71717a"
        out.append(
            '<div style="border-top:1px solid #2a2a2f;padding:10px 0">'
            '<div style="display:grid;grid-template-columns:210px 1fr 60px;gap:14px;align-items:center">'
            f'<div><div style="font-weight:600">{ICON.get(cls, "")} {html.escape(str(cls))}</div>'
            f'<div style="color:{MUTED};font-size:12px">{DESC.get(cls, "")}</div></div>'
            '<div style="height:12px;background:#2a2a2f;border-radius:6px;overflow:hidden">'
            f'<div style="width:{probs[i] * 100:.1f}%;height:12px;background:{fill};border-radius:6px"></div></div>'
            f'<div style="text-align:right;font-weight:600">{probs[i]:.0%}</div></div>'
            f'<details style="margin-top:4px"><summary style="cursor:pointer;color:{ORANGE};font-size:13px;'
            f'font-weight:600">Show why</summary>{why_panel(cls, deltas, row)}</details></div>')
    out.append("</div>")
    return '<div style="max-width:980px">' + "".join(out) + "</div>"

# ---------- curated "real examples" from the held-out test set ----------
rng_ex = np.random.default_rng(SEED)
ex_ids = []
for cls in labels:
    ex_ids += rng_ex.choice(y_test.index[(y_test == cls).to_numpy()], 3, replace=False).tolist()
disagree_ids = y_test.index[y_pred_dt != y_pred_rf].tolist()[:4]
rf_wrong_ids = y_test.index[y_pred_rf != y_test.to_numpy()].tolist()[:3]
EXAMPLES = {}
for i in dict.fromkeys(ex_ids + disagree_ids + rf_wrong_ids):
    tag = " · hard case" if i in rf_wrong_ids else (" · models disagree" if i in disagree_ids else "")
    EXAMPLES[f"{ICON.get(y_test.loc[i], '')} {y_test.loc[i]} — test row {i}{tag}"] = i
print(f"{len(EXAMPLES)} demo examples prepared from the held-out test set.")

# ---------- static preview (also a fallback if the web app cannot launch) ----------
first = next(iter(EXAMPLES.values()))
preview = header_html() + render_result(X_test.loc[[first]], "Random Forest (improved)",
                                        actual=y_test.loc[first], source=f"held-out test row {first}")
with open("outputs/demo_preview.html", "w", encoding="utf-8") as f:
    f.write(f'<html><head><meta charset="utf-8"></head><body style="background:#0f0f11;padding:24px">{preview}</body></html>')
if "ipykernel" in sys.modules:
    from IPython.display import HTML
    display(HTML(f'<div style="background:#0f0f11;padding:16px;border-radius:12px">{preview}</div>'))
else:
    print("Static preview saved -> outputs/demo_preview.html")

## 12. (Optional) Download all figures and tables for the report

In [ ]:
import shutil
shutil.make_archive("outputs", "zip", "outputs")
try:
    from google.colab import files
    files.download("outputs.zip")
except ImportError:
    print("Saved outputs.zip in the working folder.")

## 13. Launch the interactive Weather Check app
Run this cell: the app appears below the cell and Gradio prints a **public `*.gradio.live` link** (valid ~72 hours) that you can open on any device or share for the demo. Stop it with the ■ stop button.

In [ ]:
def on_classify(*args):
    *vals, model_name = args
    return render_result(make_row(vals), model_name, source="your observation")

def on_example(label, model_name):
    i = EXAMPLES[label]
    row = X_test.loc[[i]]
    vals = [None if pd.isna(v) else (float(v) if c in num_cols else v) for c, v in zip(FEATURES, row.iloc[0])]
    return [*vals, render_result(row, model_name, actual=y_test.loc[i], source=f"held-out test row {i}")]

def launch_app(share):
    import subprocess, inspect
    try:
        import gradio as gr
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
        import gradio as gr

    # Gradio moved theme/js from Blocks() to launch() in newer versions: pass them wherever they are accepted
    DARK_JS = "() => { document.body.classList.add('dark'); }"
    style = {"theme": gr.themes.Soft(primary_hue="orange", neutral_hue="zinc"), "js": DARK_JS}
    launch_params = inspect.signature(gr.Blocks.launch).parameters
    blocks_kw = {k: v for k, v in style.items() if k not in launch_params}
    launch_kw = {k: v for k, v in style.items() if k in launch_params}

    with gr.Blocks(title="Weather Check", **blocks_kw) as demo:
        gr.HTML(header_html())
        with gr.Row(equal_height=False):
            with gr.Column(scale=3):
                gr.Markdown("### 1. Enter weather observations")
                inputs = []
                with gr.Row():
                    for c in num_cols:
                        lo, hi = RANGES[c]
                        inputs.append(gr.Number(label=c, value=float(BASELINE_ROW[c]),
                                                info=f"training range {lo:g} to {hi:g}", min_width=150))
                with gr.Row():
                    for c in cat_cols:
                        inputs.append(gr.Dropdown(CATEGORIES[c], value=BASELINE_ROW[c], label=c, min_width=150))
                # put inputs back in the original feature order expected by make_row
                order = {c: i for i, c in enumerate(num_cols + cat_cols)}
                inputs = [inputs[order[c]] for c in FEATURES]
                model_choice = gr.Radio(list(MODELS), value="Random Forest (improved)", label="Model")
                classify_btn = gr.Button("Classify weather", variant="primary")
            with gr.Column(scale=2):
                gr.Markdown("### …or try a real example\nHeld-out test observations that neither model saw while "
                            "learning. *Hard case* = the Random Forest gets it wrong; *models disagree* = the two "
                            "models give different answers.")
                example_choice = gr.Dropdown(list(EXAMPLES), value=list(EXAMPLES)[0], label="Test-set example")
                example_btn = gr.Button("Analyse example", variant="primary")
        result = gr.HTML()

        classify_btn.click(on_classify, inputs=inputs + [model_choice], outputs=result)
        example_btn.click(on_example, inputs=[example_choice, model_choice], outputs=inputs + [result])
    demo.launch(share=share, debug=False, **launch_kw)
    return demo

IN_NOTEBOOK = "ipykernel" in sys.modules
if IN_NOTEBOOK or "--demo" in sys.argv:
    demo = launch_app(share=IN_NOTEBOOK)
else:
    print("Run `python weather_ml_stats.py --demo` to start the Weather Check web app locally.")